In [30]:
rules=[
    (("AND",["fever","cough"]),"flu"),
    (("AND",["flu","body_pain"]),"see_doctor"),
    (("OR",["sneezing","runny_nose"]),"cold"),
    (("AND",["fever",(("OR",["cough","sneezing"]))]),"infection")
]

facts=["fever","cough","body_pain"]

def show_condition(condition):
    if isinstance(condition,str):
        return condition
    operator,parts=condition

    return "("+(" "+operator+" ").join(show_condition(part) for part in parts)+")"


print("Facts: ",facts)
for condition,conclusion in rules:
    print("IF",show_condition(condition),"THEN",conclusion)

Facts:  ['fever', 'cough', 'body_pain']
IF (fever AND cough) THEN flu
IF (flu AND body_pain) THEN see_doctor
IF (sneezing OR runny_nose) THEN cold
IF (fever AND (cough OR sneezing)) THEN infection


In [31]:
def condition_is_true(condition,known):
    if isinstance(condition,str):
        return condition in known
    operator,parts=condition

    if operator=="AND":
        return all(condition_is_true(part,known) for part in parts)
    elif operator=="OR":
        return any(condition_is_true(part,known) for part in parts)
    return False

def forward_chaining(goal,rules,facts):
    known=list(facts)
    change=True

    while change:
        change=False
        for condition,conclusion in rules:
            if condition_is_true(condition,known) and conclusion not in known:
                known.append(conclusion)
                print("RULES FIRED: ",show_condition(condition),"-> ",conclusion)
                change=False

    print("FACTS: ",known)
    return goal in known


In [32]:
print("Goal Reached: ",forward_chaining("see_doctor",rules,facts))

RULES FIRED:  (fever AND cough) ->  flu
RULES FIRED:  (flu AND body_pain) ->  see_doctor
RULES FIRED:  (fever AND (cough OR sneezing)) ->  infection
FACTS:  ['fever', 'cough', 'body_pain', 'flu', 'see_doctor', 'infection']
Goal Reached:  True


In [33]:
def prove_condition(condition,facts,rules,checking):
    if isinstance(condition,str):
        return backward_chaining(condition,facts,rules,checking)
    operator,parts=condition

    if operator=="AND":
        return all(prove_condition(part,facts,rules,checking) for part in parts)
    elif operator=="OR":
        return any(prove_condition(part,facts,rules,checking) for part in parts)
    return False

def backward_chaining(goal,facts,rules,checking=None):
    if checking is None:
        checking=set()

    if goal in facts:
        print("Fact: ",goal)
        return True

    if goal in checking:
        return False

    checking=checking|{goal}

    for condition,conclusion in rules:
        if conclusion==goal:
            print("To prove: ",goal,", we check: ",show_condition(condition))
            if(prove_condition(condition,facts,rules,checking)):
                return True
    return False

In [34]:
print("Goal Reached: ",backward_chaining("see_doctor",facts,rules))

To prove:  see_doctor , we check:  (flu AND body_pain)
To prove:  flu , we check:  (fever AND cough)
Fact:  fever
Fact:  cough
Fact:  body_pain
Goal Reached:  True
